# More Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

from scripts.general_model import GlobalResidualRegressor

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, BayesianRidge
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_predict, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.experimental import enable_iterative_imputer  # noqa: F401"
from sklearn.impute import SimpleImputer, IterativeImputer

from xgboost import XGBRegressor

from scipy.stats import spearmanr
from scipy.optimize import minimize, minimize_scalar

/Users/junkyunglee/Codes/Co-op/proctor-prediction-challenge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Setup

In [4]:
# target variables: 'proctor_mdd_g_cm3', 'proctor_owc_pct'
df = pd.read_csv('../data/train.csv')

df['clay'] = df['psd_passing_at_0_002mm_pct']
df['silt'] = df['psd_passing_at_0_063mm_pct'] - df['psd_passing_at_0_002mm_pct']
df['sand'] = df['psd_passing_at_2mm_pct'] - df['psd_passing_at_0_063mm_pct']
df['gravel'] = 100 - df['psd_passing_at_2mm_pct']
df['fine-grained'] = (df['clay'] + df['silt'] > 15)

# Geotechnical gradation indices
df['feat_cu'] = df['psd_size_at_d60_mm'].replace(0, np.nan) / df['psd_size_at_d10_mm'].replace(0, np.nan) # Coefficient of uniformity
df['feat_cc'] = (df['psd_size_at_d30_mm'] ** 2) / (df['psd_size_at_d60_mm'].replace(0, np.nan) * df['psd_size_at_d10_mm'].replace(0, np.nan)) # Coefficient of curvature
df['feat_log_cu'] = np.log1p(df['feat_cu']) # Log-CU (skewed distribution)
df['feat_pi'] = df['atterberg_liquid_limit_pct'] - df['atterberg_plastic_limit_pct'] # Plasticity Index


features = df.columns.drop(['id'])
inputs = df.columns.drop(['id', 'proctor_mdd_g_cm3', 'proctor_owc_pct'])
outputs = ['proctor_mdd_g_cm3', 'proctor_owc_pct']

dff = df[df['fine-grained']].drop(columns=['fine-grained'])    # fine-grianed
dfc = df[~df['fine-grained']].drop(columns=['fine-grained'])   # coarse-grained

In [5]:
dff

,id,psd_size_at_d10_mm,psd_size_at_d20_mm,psd_size_at_d30_mm,psd_size_at_d40_mm,psd_size_at_d50_mm,psd_size_at_d60_mm,psd_size_at_d70_mm,psd_size_at_d80_mm,psd_size_at_d90_mm,...,atterberg_plastic_limit_pct,loss_on_ignition_pct,clay,silt,sand,gravel,feat_cu,feat_cc,feat_log_cu,feat_pi
1,1,0.014579,0.062060,0.119258,0.221877,0.423550,1.022249,2.978369,6.081199,10.622063,...,NaN,1.3,4.27,15.96,45.26,34.51,70.117909,0.954313,4.264339,NaN
2,2,0.003243,0.022301,0.087024,0.156595,0.217595,0.323793,0.512853,1.546115,10.921905,...,13.75,NaN,8.84,18.35,54.21,18.60,99.843663,7.212132,4.613571,12.70
3,3,0.000355,0.000859,0.002080,0.004772,0.009031,0.015228,0.025404,0.041920,0.151357,...,22.44,4.3,29.56,54.76,15.29,0.39,42.895775,0.800305,3.781818,25.19
4,4,0.000360,0.004868,0.010546,0.015848,0.021282,0.026753,0.033510,0.042328,0.067946,...,NaN,1.7,15.09,74.46,9.73,0.72,74.313889,11.547834,4.321665,NaN
5,5,0.019744,0.142683,0.285699,0.353478,0.437337,0.544121,0.683417,0.858374,1.209994,...,NaN,0.7,0.00,15.72,82.98,1.30,27.558803,7.597782,3.351965,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,0.000324,0.000991,0.002984,0.008524,0.022478,0.095630,0.295871,0.543019,1.510473,...,29.54,4.0,26.28,31.18,34.16,8.38,295.154321,0.287381,5.690881,30.99
191,191,0.002881,0.011201,0.025252,0.044632,0.076896,0.127275,0.168699,0.223605,0.333750,...,18.18,NaN,7.83,38.33,53.70,0.14,44.177369,1.739022,3.810596,9.76
193,193,0.002596,0.013750,0.072833,0.190724,0.312550,0.463404,0.691903,1.074778,2.969485,...,NaN,NaN,8.45,20.68,58.19,12.68,178.506934,4.409526,5.190214,NaN
199,199,0.000390,0.001069,0.002937,0.007530,0.014572,0.031049,0.105558,0.312777,0.629716,...,NaN,NaN,26.20,39.77,30.43,3.60,79.612821,0.712354,4.389658,NaN


In [8]:
df.columns

Index(['id', 'psd_size_at_d10_mm', 'psd_size_at_d20_mm', 'psd_size_at_d30_mm',
       'psd_size_at_d40_mm', 'psd_size_at_d50_mm', 'psd_size_at_d60_mm',
       'psd_size_at_d70_mm', 'psd_size_at_d80_mm', 'psd_size_at_d90_mm',
       'psd_size_at_d95_mm', 'psd_size_at_d98_mm', 'psd_has_sedimentation',
       'psd_passing_at_0_002mm_pct', 'psd_passing_at_0_063mm_pct',
       'psd_passing_at_2mm_pct', 'proctor_mdd_g_cm3', 'proctor_owc_pct',
       'proctor_diam_mm', 'grain_density_g_cm3', 'hyd_cond_kf_m_s',
       'hyd_cond_hyd_gradient', 'atterberg_liquid_limit_pct',
       'atterberg_plastic_limit_pct', 'loss_on_ignition_pct', 'clay', 'silt',
       'sand', 'gravel', 'fine-grained', 'feat_cu', 'feat_cc', 'feat_log_cu',
       'feat_pi'],
      dtype='str')

In [12]:
print(df[(df['clay'] > 0) & (df['atterberg_liquid_limit_pct'].notna())]['atterberg_liquid_limit_pct'].max())
print(df[(df['clay'] > 0) & (df['atterberg_liquid_limit_pct'].notna())]['atterberg_liquid_limit_pct'].min())

60.53
22.65


In [15]:
df['atterberg_plastic_limit_pct'].describe()

count    26.000000
mean     19.061923
std       4.305040
min      12.410000
25%      16.172500
50%      18.625000
75%      21.117500
max      29.540000
Name: atterberg_plastic_limit_pct, dtype: float64